# Import

In [1]:
# Import Modules
import pandas as pd
from google import genai
import os
import json
import time
import traceback
from dotenv import load_dotenv


# Initialize Dataset

In [2]:
# Load datasets -- raw strings fix the SyntaxWarning on Windows paths
train      = pd.read_csv(r"data\MTS-Dialog-TrainingSet28SDHP%29.csv")
validation = pd.read_csv(r"data\MTS-Dialog-Validation2029.csv")


In [3]:
# Quick look at the structure
print(train.head())
print("\nShape:", train.shape)
print("\nSection headers:", train["section_header"].unique())


   ID section_header                                       section_text  \
0   0          GENHX  Symptoms: no fever, no chills, no cough, no co...   
1   1          GENHX  Symptoms: sudden onset headache, blurry vision...   
2   2          GENHX  Symptoms: itching.\nDiagnosis: condylomas.\nHi...   
3   3    MEDICATIONS  Symptoms: N/A.\r\nDiagnosis: N/A.\r\nHistory o...   
4   4             CC  Symptoms: Burn, right arm.\r\nDiagnosis: N/A.\...   

                                            dialogue  
0  Doctor: What brings you back into the clinic t...  
1  Doctor: How're you feeling today?  \nPatient: ...  
2  Doctor: Hello, miss. What is the reason for yo...  
3  Doctor: Are you taking any over the counter me...  
4  Doctor: Hi, how are you? \nPatient: I burned m...  

Shape: (1201, 4)

Section headers: <StringArray>
[        'GENHX',   'MEDICATIONS',            'CC', 'PASTMEDICALHX',
       'ALLERGY',     'FAM/SOCHX',  'PASTSURGICAL', 'OTHER_HISTORY',
    'ASSESSMENT',           'RO

In [4]:
# IMPORTANT: The dataset is long format -- the same dialogue repeats across
# multiple rows (one per section_header). drop_duplicates gives us one row
# per unique conversation so we don't process the same dialogue many times.
unique_convos = train.drop_duplicates(subset="ID").reset_index(drop=True)
print(f"Total rows: {len(train)} -> Unique conversations: {len(unique_convos)}")


Total rows: 1201 -> Unique conversations: 1201


In [5]:
dialogue_1  = unique_convos["dialogue"][0]              # single for testing
dialogue_10 = unique_convos["dialogue"][0:10].tolist()  # first 10 for batch
print(dialogue_1)


Doctor: What brings you back into the clinic today, miss? 
Patient: I came in for a refill of my blood pressure medicine. 
Doctor: It looks like Doctor Kumar followed up with you last time regarding your hypertension, osteoarthritis, osteoporosis, hypothyroidism, allergic rhinitis and kidney stones.  Have you noticed any changes or do you have any concerns regarding these issues?  
Patient: No. 
Doctor: Have you had any fever or chills, cough, congestion, nausea, vomiting, chest pain, chest pressure?
Patient: No.  
Doctor: Great. Also, for our records, how old are you and what race do you identify yourself as?
Patient: I am seventy six years old and identify as a white female.


# Gemini LLM Setup

In [6]:
# Load API key from .env -- never hardcode it
load_dotenv()
gemini_api = os.environ.get("GEMINI_API_KEY")
print("API key loaded:", gemini_api[:10] + "..." if gemini_api else "NOT FOUND")


API key loaded: AQ.Ab8RN6L...


In [7]:
SYSTEM_PROMPT = (
    "You are a clinical documentation assistant trained to read doctor-patient\n"
    "conversation transcripts and produce structured medical records.\n"
    "\n"
    "Rules:\n"
    "- Return ONLY a valid JSON object. No explanation, no preamble, no markdown\n"
    "  code fences. The first character must be { and the last must be }.\n"
    "- Use null for any field not explicitly mentioned in the transcript.\n"
    "- Use [] for list fields where the topic was mentioned but nothing found.\n"
    "- Never invent or infer information not clearly stated.\n"
    "- For medications, capture dosage/frequency/duration exactly as spoken.\n"
    "- extraction_confidence reflects transcript clarity:\n"
    "  high = clear audio, complete sentences\n"
    "  low  = fragmented or ambiguous dialogue\n"
    "\n"
    "Return this exact JSON schema, following the SOAP medical format:\n"
    "\n"
    "{\n"
    "  \\\"summary\\\": \\\"<2-3 sentence clinical summary of the full encounter>\\\",\n"
    "  \\\"encounter_type\\\": \\\"initial_consultation | follow_up | results_review | other\\\",\n"
    "  \\\"subjective\\\": {\n"
    "    \\\"chief_complaint\\\": \\\"<patient's primary reason for the visit>\\\",\n"
    "    \\\"symptoms\\\": [\\\"<symptom>\\\"],\n"
    "    \\\"symptom_onset\\\": \\\"<when symptoms started or null>\\\",\n"
    "    \\\"medical_history\\\": [\\\"<past condition>\\\"],\n"
    "    \\\"current_medications\\\": [\\\"<current medication>\\\"],\n"
    "    \\\"allergies\\\": [\\\"<allergy>\\\"]\n"
    "  },\n"
    "  \\\"objective\\\": {\n"
    "    \\\"physical_exam_findings\\\": [\\\"<finding>\\\"],\n"
    "    \\\"vital_signs\\\": \\\"<vitals or null>\\\"\n"
    "  },\n"
    "  \\\"assessment\\\": {\n"
    "    \\\"diagnosis\\\": \\\"<diagnosis or null>\\\",\n"
    "    \\\"differential_diagnosis\\\": [\\\"<alternative>\\\"]\n"
    "  },\n"
    "  \\\"plan\\\": {\n"
    "    \\\"prescribed_medications\\\": [{\\\"name\\\": \\\"<name>\\\", \\\"dosage\\\": \\\"<dosage or null>\\\", \\\"frequency\\\": \\\"<freq or null>\\\", \\\"duration\\\": \\\"<dur or null>\\\"}],\n"
    "    \\\"treatment_plan\\\": [\\\"<instruction>\\\"],\n"
    "    \\\"investigations_ordered\\\": [\\\"<test>\\\"],\n"
    "    \\\"referrals\\\": [\\\"<specialist>\\\"],\n"
    "    \\\"follow_up\\\": \\\"<timeframe or null>\\\"\n"
    "  },\n"
    "  \\\"additional_notes\\\": \\\"<extra info or null>\\\",\n"
    "  \\\"extraction_confidence\\\": \\\"high | medium | low\\\"\n"
    "}"
).strip()


In [8]:
def build_transcript_prompt(transcript: str) -> str:
    return (
        "Extract all clinical information from the doctor-patient conversation below.\n"
        "Follow the JSON schema exactly as specified.\n\n"
        "<transcript>\n"
        + transcript.strip()
        + "\n</transcript>"
    )


# Single Dialogue Test

In [9]:
# Test on a single dialogue first before running the full batch
client = genai.Client(api_key=gemini_api)
user_prompt = build_transcript_prompt(dialogue_1)

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    config=genai.types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        max_output_tokens=1024
    ),
    contents=user_prompt
)

print(response.text)
print("\n--- Token usage ---")
print(response.usage_metadata)


{
  "summary": "A 76-year-old female presents for a medication refill for hypertension. The patient reports no new symptoms, including fever, chills, respiratory issues, or chest pain, and remains stable regarding her multiple chronic conditions.",
  "encounter_type": "follow_up",
  "subjective": {
    "chief_complaint": "Refill of blood pressure medicine",
    "symptoms": [],
    "symptom_onset": null,
    "medical_history": [
      "hypertension",
      "osteoarthritis",
      "osteoporosis",
      "hypothyroidism",
      "allergic rhinitis",
      "kidney stones"
    ],
    "current_medications": [
      "blood pressure medicine"
    ],
    "allergies": []
  },
  "objective": {
    "physical_exam_findings": [],
    "vital_signs": null
  },
  "assessment": {
    "diagnosis": "hypertension",
    "differential_diagnosis": []
  },
  "plan": {
    "prescribed_medications": [],
    "treatment_plan": [
      "Refill blood pressure medication"
    ],
    "investigations_ordered": [],
    "r

# Convert Single Response to Clean CSV

In [10]:
def flatten_lists(df):
    """
    After json_normalize(), list columns still contain actual Python lists.
    This joins them with ' | ' so the CSV doesn't get mangled.
      - Empty lists  -> None
      - Dicts inside a list (e.g. prescribed_medications) -> JSON string
    """
    df = df.copy()
    for col in df.columns:
        df[col] = df[col].apply(lambda x:
            " | ".join(
                json.dumps(i) if isinstance(i, dict) else str(i)
                for i in x
            ) if isinstance(x, list) and len(x) > 0
            else (None if isinstance(x, list) else x)
        )
    return df


In [11]:
# Step 1: Parse JSON string -> Python dict
response_dict = json.loads(response.text)

# Step 2: Flatten nested dict -> single-row DataFrame
response_df = pd.json_normalize(response_dict, sep=".")

# Step 3: Pipe-join list columns so they are human-readable in CSV
response_df_clean = flatten_lists(response_df)

# Step 4: Save
response_df_clean.to_csv("medical_report.csv", index=False)
print("Saved to medical_report.csv")

# Transposed view -- flips rows/cols so you see one field per line
print("\n--- Extracted Data (transposed for readability) ---")
print(response_df_clean.T.to_string())


Saved to medical_report.csv

--- Extracted Data (transposed for readability) ---
                                                                                                                                                                                                                                                                        0
summary                            A 76-year-old female presents for a medication refill for hypertension. The patient reports no new symptoms, including fever, chills, respiratory issues, or chest pain, and remains stable regarding her multiple chronic conditions.
encounter_type                                                                                                                                                                                                                                                  follow_up
additional_notes                                                                                                         

# Batch Processing — 10 Conversations

In [15]:
def strip_fences(text: str) -> str:
    """
    Remove markdown code fences if the model wrapped JSON in ```json...```.
    This is a defensive measure -- the prompt already tells it not to.
    """
    text = text.strip()
    if text.startswith('```'):
        lines = text.splitlines()
        text = '\n'.join(lines[1:-1]).strip()
    return text

def extract_from_dialogue(client, dialogue, verbose=False):
    """
    Runs the full extraction pipeline on one dialogue string.
    Returns a flat dict ready to be a DataFrame row, or None on failure.
    Set verbose=True to print the raw API response for debugging.
    """
    try:
        resp = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            config=genai.types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                max_output_tokens=1024
            ),
            contents=build_transcript_prompt(dialogue)
        )
        raw = resp.text
        if verbose:
            print("  RAW RESPONSE (first 300 chars):", repr(raw[:300]))
        clean = strip_fences(raw)  # strips ```json...``` if present
        result_dict = json.loads(clean)
        flat_df = pd.json_normalize(result_dict, sep=".")
        flat_df = flatten_lists(flat_df)
        return flat_df.iloc[0].to_dict()
    except Exception:
        print("  [FULL TRACEBACK]:")
        traceback.print_exc()  # shows exactly which line failed and why
        return None


## Debug Cell — Run This First!
Run this cell **before** the batch loop to see the exact error on one dialogue.

In [16]:
# Test extract_from_dialogue on just the first dialogue with verbose=True
# This will show the raw API response and any traceback so you can diagnose the issue.
test_result = extract_from_dialogue(client, dialogue_10[0], verbose=True)
if test_result:
    print("\nSUCCESS. Keys:", list(test_result.keys()))
    print("Summary:", test_result.get("summary", "N/A"))
else:
    print("\nFAILED -- see traceback above for the exact error.")


  RAW RESPONSE (first 300 chars): '{\n  "summary": "The patient, a 76-year-old female, presented for a medication refill for her hypertension. She denied any new symptoms such as fever, cough, or chest pain and reported stability regarding her chronic conditions.",\n  "encounter_type": "follow_up",\n  "subjective": {\n    "chief_complain'

SUCCESS. Keys: ['summary', 'encounter_type', 'additional_notes', 'extraction_confidence', 'subjective.chief_complaint', 'subjective.symptoms', 'subjective.symptom_onset', 'subjective.medical_history', 'subjective.current_medications', 'subjective.allergies', 'objective.physical_exam_findings', 'objective.vital_signs', 'assessment.diagnosis', 'assessment.differential_diagnosis', 'plan.prescribed_medications', 'plan.treatment_plan', 'plan.investigations_ordered', 'plan.referrals', 'plan.follow_up']
Summary: The patient, a 76-year-old female, presented for a medication refill for her hypertension. She denied any new symptoms such as fever, cough, or che

In [17]:
rows   = []
failed = []

for i, dialogue in enumerate(dialogue_10):
    print(f"Processing conversation {i+1}/10...", end=" ")
    result = extract_from_dialogue(client, dialogue)
    if result:
        result["conversation_id"] = i
        rows.append(result)
        print("OK")
    else:
        failed.append(i)
        print("FAILED")
    time.sleep(1)  # stay within free-tier rate limits

print(f"\nDone. Successful: {len(rows)}/10 | Failed: {len(failed)}")
if failed:
    print(f"Failed conversation indices: {failed}")


Processing conversation 1/10... OK
Processing conversation 2/10... OK
Processing conversation 3/10... OK
Processing conversation 4/10... OK
Processing conversation 5/10... OK
Processing conversation 6/10... OK
Processing conversation 7/10... OK
Processing conversation 8/10... OK
Processing conversation 9/10... OK
Processing conversation 10/10... OK

Done. Successful: 10/10 | Failed: 0


In [18]:
# Combine all rows into one DataFrame
batch_df = pd.DataFrame(rows)

# Put conversation_id as the first column
cols = ["conversation_id"] + [c for c in batch_df.columns if c != "conversation_id"]
batch_df = batch_df[cols]

batch_df.to_csv("medical_report_batch.csv", index=False)
print(f"Saved {len(batch_df)} records to medical_report_batch.csv")
print(f"\nColumns ({len(batch_df.columns)}): {list(batch_df.columns)}")
print("\nFirst record (transposed):")
print(batch_df.iloc[0].to_string())


Saved 10 records to medical_report_batch.csv

Columns (20): ['conversation_id', 'summary', 'encounter_type', 'additional_notes', 'extraction_confidence', 'subjective.chief_complaint', 'subjective.symptoms', 'subjective.symptom_onset', 'subjective.medical_history', 'subjective.current_medications', 'subjective.allergies', 'objective.physical_exam_findings', 'objective.vital_signs', 'assessment.diagnosis', 'assessment.differential_diagnosis', 'plan.prescribed_medications', 'plan.treatment_plan', 'plan.investigations_ordered', 'plan.referrals', 'plan.follow_up']

First record (transposed):
conversation_id                                                                      0
summary                              The patient is a 76-year-old female presenting...
encounter_type                                                               follow_up
additional_notes                     Patient is 76 years old and identifies as a wh...
extraction_confidence                                     